# IITM DL & GenAI Project: Smart MCQ Solver

**Objective**: The goal of this competition is to build a machine learning pipeline capable of answering complex multiple-choice questions. 

Given a question prompt and five possible options (A, B, C, D, E), the model must rank the top 3 most likely answers. The evaluation metric for the competition is **Mean Average Precision at 3 (MAP@3)**, meaning models are rewarded heavily for getting the correct answer as their #1 prediction.

---

## Section 1: Project Setup & Data Ingestion
*Initializing the environment and loading the competition datasets.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load datasets (Using Kaggle Environment Paths)
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

# Fallback for local testing if running outside Kaggle
if not os.path.exists(train_path):
    train_path = 'train.csv'
    test_path = 'test.csv'

# Explicitly cast text columns to string to prevent NaN parsing issues
dtype_dict = {'prompt': str, 'A': str, 'B': str, 'C': str, 'D': str, 'E': str, 'answer': str}
train_df = pd.read_csv(train_path, dtype=dtype_dict)
test_df = pd.read_csv(test_path, dtype=dtype_dict)

# Force ID column to string just in case
if 'id' in train_df.columns:
    train_df['id'] = train_df['id'].astype(str)
if 'id' in test_df.columns:
    test_df['id'] = test_df['id'].astype(str)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}")
print("\nData Types:")
print(train_df.dtypes)

## Section 2: Global Data Cleaning
*Handling missing values and duplicates across the entire dataset before applying any model-specific logic.*

In [ ]:
# 1. Missing value analysis
print("Missing Values Before Cleaning:\n", train_df.isnull().sum())

# Fill missing text with empty strings universally (avoids float/NaN errors in transformers)
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
train_df[text_cols] = train_df[text_cols].fillna("")
test_df[text_cols] = test_df[text_cols].fillna("")

In [ ]:
# 2. Duplicate Check
duplicates = train_df.duplicated(subset=['prompt']).sum()
print(f"Found {duplicates} duplicate prompts in training set. Keeping all for now as options might differ.")

## Section 3: Exploratory Data Analysis (EDA)
*Visualizing class distributions, text lengths, and option variances.*

In [ ]:
sns.set_theme(style="whitegrid")

# Plot 1: Answer Distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x='answer', order=['A', 'B', 'C', 'D', 'E'], palette='viridis')
plt.title('Target Class Distribution')
plt.show()

In [ ]:
# Plot 2: Prompt Word Count Distribution (KDE)
train_df['prompt_word_count'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 4))
sns.histplot(data=train_df, x='prompt_word_count', bins=30, kde=True, color='blue')
plt.title('Prompt Length (Word Count) Distribution')
plt.show()

In [ ]:
# Plot 3: Option Lengths (A vs B vs C vs D vs E)
option_lengths = []
for opt in ['A', 'B', 'C', 'D', 'E']:
    lengths = train_df[opt].apply(lambda x: len(str(x).split()))
    for l in lengths:
        option_lengths.append({'Option': opt, 'Word Count': l})
opt_df = pd.DataFrame(option_lengths)

plt.figure(figsize=(8, 4))
sns.boxplot(data=opt_df, x='Option', y='Word Count', palette='pastel')
plt.title('Word Counts Across Options')
plt.ylim(0, 50) # Capped for readability
plt.show()

In [ ]:
# Plot 4: Does prompt length affect the correct answer?
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_df, x='answer', y='prompt_word_count', order=['A', 'B', 'C', 'D', 'E'], palette='muted')
plt.title('Prompt Length vs. Correct Answer')
plt.show()

## Section 4: Shared Utilities
*Foundational helper functions for evaluation and validation splits.*

### Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Create reproducible validation split
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])
print(f"Train Split: {len(train_split)} rows")
print(f"Validation Split: {len(val_split)} rows")

### Evaluation Metric (MAP@3)

In [ ]:
def calculate_map3(predictions, ground_truth):
    """Calculates Mean Average Precision @ 3."""
    scores = []
    for pred_str, gt in zip(predictions, ground_truth):
        pred_list = pred_str.strip().split()
        score = 0.0
        num_correct = 0
        for k, label in enumerate(pred_list[:3], start=1):
            if label == gt:
                num_correct += 1
                score += num_correct / k
        scores.append(score)
    return np.mean(scores)

# Quick sanity check to ensure the metric works as expected
dummy_preds = ['A B C', 'D A E', 'C B A']
dummy_truths = ['A', 'A', 'C']
print(f"Sanity Check MAP@3 Score: {calculate_map3(dummy_preds, dummy_truths)}")

## Section 5: Model 1 (TF-IDF Baseline Preparation)
*Unlike Transformer models which rely on case and punctuation for semantic context, TF-IDF is a lexical Bag-of-Words model. We must aggressively normalize the text specifically for this model.*

In [ ]:
def clean_text_for_tfidf(text):
    """Aggressive cleaning strictly for TF-IDF (lowercasing, removing punctuation)."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Note: We might need to add stopword removal here later if results are poor
    return text.strip()

print("Original:", train_df['prompt'].iloc[0][:100])
print("Cleaned TF-IDF:", clean_text_for_tfidf(train_df['prompt'].iloc[0][:100]))